In [67]:
import pandas as pd
import numpy as np

# Load cleaned datasets
users = pd.read_csv("../data/processed/users_clean.csv")
creators = pd.read_csv("../data/processed/creators_clean.csv")
content = pd.read_csv("../data/processed/content_clean.csv")
interactions = pd.read_csv("../data/processed/interactions_clean.csv")

# Convert timestamp columns
users["signup_date"] = pd.to_datetime(users["signup_date"])
content["created_at"] = pd.to_datetime(content["created_at"])
interactions["timestamp"] = pd.to_datetime(interactions["timestamp"])

print("Users:", users.shape)
print("Creators:", creators.shape)
print("Content:", content.shape)
print("Interactions:", interactions.shape)

print("\nInteraction columns:")
print(interactions.columns.tolist())

Users: (5000, 7)
Creators: (500, 5)
Content: (10000, 9)
Interactions: (250000, 22)

Interaction columns:
['user_id', 'content_id', 'creator_id', 'genre', 'content_type', 'duration', 'creator_followers', 'content_created_at', 'genre_match', 'timestamp', 'content_age_days', 'freshness', 'impression', 'clicked', 'watch_time', 'completion_rate', 'liked', 'saved', 'shared', 'commented', 'recreated', 'meaningful_engagement']


In [68]:
print("Interaction period:")
print("Start:", interactions["timestamp"].min())
print("End:", interactions["timestamp"].max())

print("\nContent creation period:")
print("Start:", content["created_at"].min())
print("End:", content["created_at"].max())

Interaction period:
Start: 2025-06-01 23:55:16
End: 2026-08-30 23:59:46

Content creation period:
Start: 2025-06-01 03:58:39
End: 2026-08-30 23:55:53


In [69]:
interactions = interactions.sort_values("timestamp").reset_index(drop=True)

interactions.head()

,user_id,content_id,creator_id,genre,content_type,duration,creator_followers,content_created_at,genre_match,timestamp,...,impression,clicked,watch_time,completion_rate,liked,saved,shared,commented,recreated,meaningful_engagement
0,U04449,CT006574,C0314,Comedy,cine,30.4,101,2025-06-01 14:38:24,False,2025-06-01 23:55:16,...,1,0,0.00,0.000000,0,0,0,0,0,0
1,U01229,CT006567,C0240,Comedy,cine,35.9,40,2025-06-01 08:55:23,False,2025-06-02 00:20:44,...,1,0,0.00,0.000000,0,0,0,0,0,0
2,U03221,CT009509,C0196,Romance,cine,37.9,10,2025-06-02 05:26:06,True,2025-06-02 13:22:49,...,1,1,31.65,0.835092,0,1,1,0,0,1
3,U01150,CT000021,C0051,Drama,mini,12.9,60,2025-06-02 01:00:37,False,2025-06-03 13:33:04,...,1,0,0.00,0.000000,0,0,0,0,0,0
4,U00653,CT001952,C0100,Action,video,26.6,10,2025-06-03 16:54:12,True,2025-06-04 00:38:55,...,1,1,12.90,0.484962,0,0,0,0,0,1


In [70]:
# Check whether any interaction happened before the corresponding content was created

temporal_check = interactions["timestamp"] < interactions["content_created_at"]

print("Interactions before content creation:", temporal_check.sum())
print(
    "Percentage:",
    round(temporal_check.mean() * 100, 4),
    "%"
)

Interactions before content creation: 0
Percentage: 0.0 %


## 1. Time-Aware Train / Validation / Test Split

Because user behavior and content evolve over time, a random split
could introduce temporal leakage.

We therefore split interactions chronologically:

- 70% earliest interactions → Training set
- 15% subsequent interactions → Validation set
- 15% latest interactions → Test set

This simulates the real-world scenario of training on historical behavior
and predicting future engagement.

In [71]:
# Ensure interactions are sorted chronologically
interactions = interactions.sort_values("timestamp").reset_index(drop=True)

n = len(interactions)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = interactions.iloc[:train_end].copy()
validation = interactions.iloc[train_end:val_end].copy()
test = interactions.iloc[val_end:].copy()

print("Dataset sizes:")
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print("\nTime ranges:")
print(
    "Train:",
    train["timestamp"].min(),
    "→",
    train["timestamp"].max()
)

print(
    "Validation:",
    validation["timestamp"].min(),
    "→",
    validation["timestamp"].max()
)

print(
    "Test:",
    test["timestamp"].min(),
    "→",
    test["timestamp"].max()
)

Dataset sizes:
Train: (175000, 22)
Validation: (37500, 22)
Test: (37500, 22)

Time ranges:
Train: 2025-06-01 23:55:16 → 2026-07-22 07:49:04
Validation: 2026-07-22 07:50:29 → 2026-08-15 05:05:32
Test: 2026-08-15 05:05:41 → 2026-08-30 23:59:46


In [72]:
# Verify that the temporal split has no overlap

assert train["timestamp"].max() < validation["timestamp"].min()
assert validation["timestamp"].max() < test["timestamp"].min()

print("✓ Temporal ordering verified")
print("✓ No overlap between train, validation, and test periods")

✓ Temporal ordering verified
✓ No overlap between train, validation, and test periods


## 2. User-Level Historical Features

User behavior is summarized from historical interactions.

To avoid target leakage, these features are calculated using
training-period interactions only.

The resulting user profile represents what the system would have
known about each user before making predictions on later interactions.

In [73]:
# Build historical user-level features from training data only

user_features = (
    train
    .groupby("user_id")
    .agg(
        user_total_interactions=("user_id", "count"),
        user_click_rate=("clicked", "mean"),
        user_avg_watch_time=("watch_time", "mean"),
        user_avg_completion=("completion_rate", "mean"),
        user_like_rate=("liked", "mean"),
        user_save_rate=("saved", "mean"),
        user_share_rate=("shared", "mean"),
        user_comment_rate=("commented", "mean"),
        user_recreate_rate=("recreated", "mean"),
    )
    .reset_index()
)

print("User feature shape:", user_features.shape)

user_features.head()

User feature shape: (4996, 10)


,user_id,user_total_interactions,user_click_rate,user_avg_watch_time,user_avg_completion,user_like_rate,user_save_rate,user_share_rate,user_comment_rate,user_recreate_rate
0,U00001,37,0.405405,12.772432,0.309512,0.135135,0.027027,0.000000,0.0,0.000000
1,U00002,8,0.250000,5.118750,0.219667,0.000000,0.000000,0.000000,0.0,0.000000
2,U00003,49,0.265306,9.512653,0.189347,0.102041,0.020408,0.020408,0.0,0.040816
3,U00004,54,0.148148,3.719074,0.100005,0.055556,0.000000,0.018519,0.0,0.018519
4,U00006,5,0.200000,2.976000,0.174035,0.200000,0.200000,0.200000,0.0,0.000000


## 3. Content-Level Features

Content-level features describe properties that are available when
content is published or presented to a user.

These include:

- Content duration
- Genre
- Content type
- Creator follower count
- Content age
- Freshness

Static content attributes do not introduce temporal leakage because
they are known independently of future user interactions.

In [74]:
# Build content-level features

content_features = content[
    [
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration"
    ]
].copy()

# Add creator follower information
content_features = content_features.merge(
    creators[
        ["creator_id", "followers"]
    ],
    on="creator_id",
    how="left"
)

# Rename for clarity
content_features = content_features.rename(
    columns={
        "followers": "creator_followers"
    }
)

print("Content feature shape:", content_features.shape)

content_features.head()

Content feature shape: (10000, 6)


,content_id,creator_id,genre,content_type,duration,creator_followers
0,CT000001,C0131,Mystery,mini,24.6,715
1,CT000002,C0484,Horror,mini,21.6,291
2,CT000003,C0195,Sci-Fi,mini,14.7,33
3,CT000004,C0328,Romance,mini,6.6,49
4,CT000005,C0406,Action,image,88.7,74


## 4. Temporal Content Features

Content freshness changes over time.

For each interaction, we use:

- `content_age_days`: age of the content when the interaction occurred
- `freshness`: exponentially decayed freshness score

These features are point-in-time valid because they depend only on
the content creation timestamp and the interaction timestamp.

In [75]:
# Create interaction-level temporal features

temporal_features = interactions[
    [
        "user_id",
        "content_id",
        "timestamp",
        "content_age_days",
        "freshness"
    ]
].copy()

print("Temporal feature shape:", temporal_features.shape)

temporal_features.head()

Temporal feature shape: (250000, 5)


,user_id,content_id,timestamp,content_age_days,freshness
0,U04449,CT006574,2025-06-01 23:55:16,0.386713,0.987192
1,U01229,CT006567,2025-06-02 00:20:44,0.642604,0.978808
2,U03221,CT009509,2025-06-02 13:22:49,0.331053,0.989026
3,U01150,CT000021,2025-06-03 13:33:04,1.522535,0.950515
4,U00653,CT001952,2025-06-04 00:38:55,0.322720,0.989300


## 5. Historical Content Performance

Content performance features summarize historical behavior observed
before the prediction period.

These features include:

- Prior impressions
- Prior clicks
- Prior meaningful engagements
- Historical click-through rate
- Historical engagement rate

Only training-period interactions are used to construct these features.

This prevents future interaction outcomes from leaking into the model.

In [76]:
# Historical content performance from training data only

content_history = (
    train
    .groupby("content_id")
    .agg(
        content_prior_impressions=("impression", "sum"),
        content_prior_clicks=("clicked", "sum"),
        content_prior_engagements=("meaningful_engagement", "sum"),
    )
    .reset_index()
)

# Calculate historical rates
content_history["content_prior_ctr"] = (
    content_history["content_prior_clicks"]
    / content_history["content_prior_impressions"]
)

content_history["content_prior_engagement_rate"] = (
    content_history["content_prior_engagements"]
    / content_history["content_prior_impressions"]
)

print("Historical content feature shape:", content_history.shape)

content_history.head()

Historical content feature shape: (9103, 6)


,content_id,content_prior_impressions,content_prior_clicks,content_prior_engagements,content_prior_ctr,content_prior_engagement_rate
0,CT000001,28,13,13,0.464286,0.464286
1,CT000003,4,3,3,0.750000,0.750000
2,CT000004,14,5,5,0.357143,0.357143
3,CT000005,17,7,7,0.411765,0.411765
4,CT000006,21,9,9,0.428571,0.428571


In [77]:
# Validate historical content features

print("Missing values:")
print(content_history.isna().sum())

print("\nZero-impression content:")
print(
    (content_history["content_prior_impressions"] == 0).sum()
)

print("\nHistorical CTR range:")
print(
    content_history["content_prior_ctr"].min(),
    "→",
    content_history["content_prior_ctr"].max()
)

print("\nHistorical engagement rate range:")
print(
    content_history["content_prior_engagement_rate"].min(),
    "→",
    content_history["content_prior_engagement_rate"].max()
)

Missing values:
content_id                       0
content_prior_impressions        0
content_prior_clicks             0
content_prior_engagements        0
content_prior_ctr                0
content_prior_engagement_rate    0
dtype: int64

Zero-impression content:
0

Historical CTR range:
0.0 → 1.0

Historical engagement rate range:
0.0 → 1.0


## 6. Combined Content Features

Static content attributes are combined with historical performance
features.

Content with no historical training interactions will retain missing
historical-performance values after the merge. These represent
cold-start content and will be handled explicitly rather than
imputing artificial performance.

In [78]:
# Combine static and historical content features

content_features = content_features.merge(
    content_history,
    on="content_id",
    how="left"
)

print("Combined content feature shape:", content_features.shape)

print("\nMissing historical features:")
print(
    content_features[
        [
            "content_prior_impressions",
            "content_prior_clicks",
            "content_prior_engagements",
            "content_prior_ctr",
            "content_prior_engagement_rate"
        ]
    ].isna().sum()
)

Combined content feature shape: (10000, 11)

Missing historical features:
content_prior_impressions        897
content_prior_clicks             897
content_prior_engagements        897
content_prior_ctr                897
content_prior_engagement_rate    897
dtype: int64


## 7. Point-in-Time User History

Historical user behavior must be calculated using only interactions
that occurred before the current interaction.

For each interaction, we calculate:

- Number of previous interactions by the user

The current interaction is excluded from its own history.

This prevents future information from leaking into model features.

In [79]:
# Make a working copy of the chronologically ordered interactions

model_data = interactions.sort_values(
    ["user_id", "timestamp"]
).copy()

# Number of previous interactions for each user
# shift(1) ensures the current interaction is excluded

model_data["user_prior_interactions"] = (
    model_data
    .groupby("user_id")
    .cumcount()
)

print(
    model_data[
        [
            "user_id",
            "timestamp",
            "user_prior_interactions",
            "meaningful_engagement"
        ]
    ].head(15)
)

      user_id           timestamp  user_prior_interactions  \
5027   U00001 2025-08-29 04:40:07                        0   
13463  U00001 2025-10-20 17:40:08                        1   
15271  U00001 2025-10-29 11:12:49                        2   
21293  U00001 2025-11-23 05:33:29                        3   
25966  U00001 2025-12-09 15:59:55                        4   
29179  U00001 2025-12-19 22:27:29                        5   
36454  U00001 2026-01-10 00:24:10                        6   
39062  U00001 2026-01-17 00:08:40                        7   
50469  U00001 2026-02-13 14:19:29                        8   
50696  U00001 2026-02-14 02:30:09                        9   
52263  U00001 2026-02-17 13:00:08                       10   
52371  U00001 2026-02-17 18:41:53                       11   
54020  U00001 2026-02-21 06:11:45                       12   
54362  U00001 2026-02-22 00:29:04                       13   
56270  U00001 2026-02-25 21:08:33                       14   

       

## 8. Point-in-Time User Engagement Features

Historical user behavior is calculated cumulatively over time.

For each interaction, only behavior from previous interactions is used.

The current interaction is excluded from all historical aggregates.

This produces features such as historical click rate, engagement rate,
average watch time, and interaction history without target leakage.

In [80]:
# Sort by user and timestamp
model_data = model_data.sort_values(
    ["user_id", "timestamp"]
).copy()

# Historical cumulative counts and sums
user_group = model_data.groupby("user_id")

model_data["user_prior_clicks"] = (
    user_group["clicked"].cumsum() - model_data["clicked"]
)

model_data["user_prior_engagements"] = (
    user_group["meaningful_engagement"].cumsum()
    - model_data["meaningful_engagement"]
)

model_data["user_prior_watch_time"] = (
    user_group["watch_time"].cumsum()
    - model_data["watch_time"]
)

model_data["user_prior_completions"] = (
    user_group["completion_rate"].cumsum()
    - model_data["completion_rate"]
)

# Historical interaction count
model_data["user_prior_interactions"] = (
    user_group.cumcount()
)

# Historical rates
model_data["user_prior_ctr"] = (
    model_data["user_prior_clicks"]
    / model_data["user_prior_interactions"].replace(0, np.nan)
)

model_data["user_prior_engagement_rate"] = (
    model_data["user_prior_engagements"]
    / model_data["user_prior_interactions"].replace(0, np.nan)
)

model_data["user_prior_avg_watch_time"] = (
    model_data["user_prior_watch_time"]
    / model_data["user_prior_interactions"].replace(0, np.nan)
)

model_data["user_prior_avg_completion"] = (
    model_data["user_prior_completions"]
    / model_data["user_prior_interactions"].replace(0, np.nan)
)

print(
    model_data[
        [
            "user_id",
            "timestamp",
            "user_prior_interactions",
            "user_prior_clicks",
            "user_prior_engagements",
            "user_prior_ctr",
            "user_prior_engagement_rate",
            "user_prior_avg_watch_time",
            "user_prior_avg_completion"
        ]
    ].head(15)
)

      user_id           timestamp  user_prior_interactions  user_prior_clicks  \
5027   U00001 2025-08-29 04:40:07                        0                  0   
13463  U00001 2025-10-20 17:40:08                        1                  0   
15271  U00001 2025-10-29 11:12:49                        2                  0   
21293  U00001 2025-11-23 05:33:29                        3                  0   
25966  U00001 2025-12-09 15:59:55                        4                  1   
29179  U00001 2025-12-19 22:27:29                        5                  1   
36454  U00001 2026-01-10 00:24:10                        6                  1   
39062  U00001 2026-01-17 00:08:40                        7                  1   
50469  U00001 2026-02-13 14:19:29                        8                  2   
50696  U00001 2026-02-14 02:30:09                        9                  2   
52263  U00001 2026-02-17 13:00:08                       10                  2   
52371  U00001 2026-02-17 18:

## 9. Point-in-Time Creator History

Creator-level performance is calculated using only interactions that
occurred before the current interaction.

These features capture the historical performance of a creator while
avoiding information leakage from future interactions.

In [81]:
# Sort chronologically within each creator
model_data = model_data.sort_values(
    ["creator_id", "timestamp"]
).copy()

creator_group = model_data.groupby("creator_id")

# Historical creator metrics
model_data["creator_prior_interactions"] = (
    creator_group.cumcount()
)

model_data["creator_prior_clicks"] = (
    creator_group["clicked"].cumsum()
    - model_data["clicked"]
)

model_data["creator_prior_engagements"] = (
    creator_group["meaningful_engagement"].cumsum()
    - model_data["meaningful_engagement"]
)

# Historical rates
model_data["creator_prior_ctr"] = (
    model_data["creator_prior_clicks"]
    / model_data["creator_prior_interactions"].replace(0, np.nan)
)

model_data["creator_prior_engagement_rate"] = (
    model_data["creator_prior_engagements"]
    / model_data["creator_prior_interactions"].replace(0, np.nan)
)

print(
    model_data[
        [
            "creator_id",
            "timestamp",
            "creator_prior_interactions",
            "creator_prior_clicks",
            "creator_prior_engagements",
            "creator_prior_ctr",
            "creator_prior_engagement_rate"
        ]
    ].head(15)
)

     creator_id           timestamp  creator_prior_interactions  \
956       C0001 2025-07-11 15:03:16                           0   
2070      C0001 2025-07-29 11:06:24                           1   
2099      C0001 2025-07-29 18:24:38                           2   
2789      C0001 2025-08-07 21:10:35                           3   
3214      C0001 2025-08-12 10:10:56                           4   
3552      C0001 2025-08-15 22:25:45                           5   
3878      C0001 2025-08-19 05:13:33                           6   
3947      C0001 2025-08-19 20:36:21                           7   
4578      C0001 2025-08-25 06:04:19                           8   
4721      C0001 2025-08-26 13:44:24                           9   
4874      C0001 2025-08-27 19:31:19                          10   
5526      C0001 2025-09-02 11:59:24                          11   
5625      C0001 2025-09-03 06:23:36                          12   
5700      C0001 2025-09-03 19:15:23                          1

## 10. Point-in-Time Content History

Content performance evolves as users interact with content.

For each interaction, historical content metrics are calculated using
only previous interactions with that content.

This prevents future content performance from leaking into the
current prediction.

In [82]:
# Sort chronologically within each content item
model_data = model_data.sort_values(
    ["content_id", "timestamp"]
).copy()

content_group = model_data.groupby("content_id")

# Historical content interaction count
model_data["content_prior_interactions"] = (
    content_group.cumcount()
)

# Historical clicks
model_data["content_prior_clicks"] = (
    content_group["clicked"].cumsum()
    - model_data["clicked"]
)

# Historical meaningful engagements
model_data["content_prior_engagements"] = (
    content_group["meaningful_engagement"].cumsum()
    - model_data["meaningful_engagement"]
)

# Historical CTR
model_data["content_prior_ctr"] = (
    model_data["content_prior_clicks"]
    / model_data["content_prior_interactions"].replace(0, np.nan)
)

# Historical engagement rate
model_data["content_prior_engagement_rate"] = (
    model_data["content_prior_engagements"]
    / model_data["content_prior_interactions"].replace(0, np.nan)
)

print(
    model_data[
        [
            "content_id",
            "timestamp",
            "content_prior_interactions",
            "content_prior_clicks",
            "content_prior_engagements",
            "content_prior_ctr",
            "content_prior_engagement_rate"
        ]
    ].head(15)
)

      content_id           timestamp  content_prior_interactions  \
31704   CT000001 2025-12-27 17:48:45                           0   
36813   CT000001 2026-01-10 23:29:55                           1   
41813   CT000001 2026-01-23 23:32:29                           2   
48685   CT000001 2026-02-09 15:24:55                           3   
50068   CT000001 2026-02-12 17:46:20                           4   
50516   CT000001 2026-02-13 16:43:03                           5   
51481   CT000001 2026-02-15 22:21:22                           6   
53761   CT000001 2026-02-20 16:10:48                           7   
60284   CT000001 2026-03-05 23:17:12                           8   
64975   CT000001 2026-03-14 22:55:41                           9   
69988   CT000001 2026-03-24 02:02:49                          10   
72137   CT000001 2026-03-27 20:42:48                          11   
72153   CT000001 2026-03-27 21:17:10                          12   
73429   CT000001 2026-03-30 02:21:26            

## 11. Point-in-Time User-Content History

User-content history captures the previous relationship between a user
and a specific content item.

These features help distinguish first-time exposure from repeated
interaction while using only information available before the current
interaction.

In [83]:
# Sort chronologically within each user-content pair
model_data = model_data.sort_values(
    ["user_id", "content_id", "timestamp"]
).copy()

user_content_group = model_data.groupby(
    ["user_id", "content_id"]
)

# Historical interaction count
model_data["user_content_prior_interactions"] = (
    user_content_group.cumcount()
)

# Historical clicks
model_data["user_content_prior_clicks"] = (
    user_content_group["clicked"].cumsum()
    - model_data["clicked"]
)

# Historical meaningful engagements
model_data["user_content_prior_engagements"] = (
    user_content_group["meaningful_engagement"].cumsum()
    - model_data["meaningful_engagement"]
)

# Historical CTR
model_data["user_content_prior_ctr"] = (
    model_data["user_content_prior_clicks"]
    / model_data["user_content_prior_interactions"].replace(0, np.nan)
)

# Historical engagement rate
model_data["user_content_prior_engagement_rate"] = (
    model_data["user_content_prior_engagements"]
    / model_data["user_content_prior_interactions"].replace(0, np.nan)
)

print(
    model_data[
        [
            "user_id",
            "content_id",
            "timestamp",
            "user_content_prior_interactions",
            "user_content_prior_clicks",
            "user_content_prior_engagements",
            "user_content_prior_ctr",
            "user_content_prior_engagement_rate"
        ]
    ].head(20)
)

       user_id content_id           timestamp  \
197819  U00001   CT000042 2026-08-06 14:53:21   
25966   U00001   CT000240 2025-12-09 15:59:55   
193243  U00001   CT000275 2026-08-03 16:57:04   
181998  U00001   CT000303 2026-07-27 04:36:26   
74322   U00001   CT001189 2026-03-31 12:12:06   
211147  U00001   CT001286 2026-08-14 10:59:57   
50469   U00001   CT001594 2026-02-13 14:19:29   
109854  U00001   CT001811 2026-05-20 20:04:01   
197228  U00001   CT001900 2026-08-06 06:00:16   
120614  U00001   CT001909 2026-06-02 10:14:28   
71585   U00001   CT001989 2026-03-26 20:11:32   
56270   U00001   CT002240 2026-02-25 21:08:33   
193340  U00001   CT002495 2026-08-03 18:42:01   
89456   U00001   CT002570 2026-04-24 04:34:29   
105530  U00001   CT002570 2026-05-15 15:25:11   
152450  U00001   CT002627 2026-07-03 22:53:16   
185597  U00001   CT002641 2026-07-29 15:13:33   
222786  U00001   CT002708 2026-08-20 13:07:51   
5027    U00001   CT003005 2025-08-29 04:40:07   
194984  U00001   CT0

## 12. Point-in-Time User Genre Affinity

User genre affinity captures a user's historical interaction and
engagement behavior within each genre.

The features are calculated using only interactions that occurred
before the current interaction, preventing target leakage.

In [84]:
# Sort chronologically within each user and genre
model_data = model_data.sort_values(
    ["user_id", "genre", "timestamp"]
).copy()

user_genre_group = model_data.groupby(
    ["user_id", "genre"]
)

# Historical interactions within the genre
model_data["user_genre_prior_interactions"] = (
    user_genre_group.cumcount()
)

# Historical clicks within the genre
model_data["user_genre_prior_clicks"] = (
    user_genre_group["clicked"].cumsum()
    - model_data["clicked"]
)

# Historical meaningful engagements within the genre
model_data["user_genre_prior_engagements"] = (
    user_genre_group["meaningful_engagement"].cumsum()
    - model_data["meaningful_engagement"]
)

# Historical genre engagement rate
model_data["user_genre_prior_engagement_rate"] = (
    model_data["user_genre_prior_engagements"]
    / model_data["user_genre_prior_interactions"].replace(0, np.nan)
)

print(
    model_data[
        [
            "user_id",
            "genre",
            "timestamp",
            "user_genre_prior_interactions",
            "user_genre_prior_clicks",
            "user_genre_prior_engagements",
            "user_genre_prior_engagement_rate"
        ]
    ].head(20)
)

       user_id        genre           timestamp  \
39062   U00001       Action 2026-01-17 00:08:40   
74322   U00001       Action 2026-03-31 12:12:06   
184178  U00001       Action 2026-07-28 16:20:51   
36454   U00001    Animation 2026-01-10 00:24:10   
86215   U00001    Animation 2026-04-19 08:55:48   
120614  U00001    Animation 2026-06-02 10:14:28   
144647  U00001    Animation 2026-06-26 21:07:17   
158695  U00001    Animation 2026-07-09 08:48:05   
184775  U00001    Animation 2026-07-29 02:24:35   
189272  U00001    Animation 2026-08-01 01:24:04   
214602  U00001    Animation 2026-08-16 08:07:24   
56270   U00001       Comedy 2026-02-25 21:08:33   
193243  U00001       Comedy 2026-08-03 16:57:04   
197228  U00001       Comedy 2026-08-06 06:00:16   
207060  U00001       Comedy 2026-08-12 03:53:12   
224236  U00001       Comedy 2026-08-21 05:30:17   
241030  U00001       Comedy 2026-08-28 07:31:59   
13463   U00001  Documentary 2025-10-20 17:40:08   
54362   U00001  Documentary 202

## 13. Temporal and Context Features

Temporal features capture the context in which an interaction occurs.

These features are available at prediction time and include time of day,
day of week, weekend behavior, content freshness, and user tenure.

They help the model learn differences in engagement behavior across
different usage contexts.

In [88]:
# Restore global chronological order before creating
# interaction-level temporal features
model_data = model_data.sort_values("timestamp").copy()

# Time of interaction
model_data["interaction_hour"] = (
    model_data["timestamp"].dt.hour
)

model_data["interaction_day_of_week"] = (
    model_data["timestamp"].dt.dayofweek
)

model_data["is_weekend"] = (
    model_data["interaction_day_of_week"] >= 5
).astype(int)

# Cyclical representation of hour
model_data["hour_sin"] = np.sin(
    2 * np.pi * model_data["interaction_hour"] / 24
)

model_data["hour_cos"] = np.cos(
    2 * np.pi * model_data["interaction_hour"] / 24
)

# User tenure at the time of interaction
model_data = model_data.merge(
    users[["user_id", "signup_date"]],
    on="user_id",
    how="left"
)

model_data["signup_date"] = pd.to_datetime(
    model_data["signup_date"]
)

model_data["user_tenure_days"] = (
    model_data["timestamp"]
    - model_data["signup_date"]
).dt.total_seconds() / (24 * 60 * 60)

# Verify temporal/context features
print(
    model_data[
        [
            "timestamp",
            "interaction_hour",
            "interaction_day_of_week",
            "is_weekend",
            "hour_sin",
            "hour_cos",
            "user_tenure_days",
            "content_age_days",
            "freshness"
        ]
    ].head(10)
)

            timestamp  interaction_hour  interaction_day_of_week  is_weekend  \
0 2025-06-01 23:55:16                23                        6           1   
1 2025-06-02 00:20:44                 0                        0           0   
2 2025-06-02 13:22:49                13                        0           0   
3 2025-06-03 13:33:04                13                        1           0   
4 2025-06-04 00:38:55                 0                        2           0   
5 2025-06-04 03:51:02                 3                        2           0   
6 2025-06-04 10:08:07                10                        2           0   
7 2025-06-04 14:13:45                14                        2           0   
8 2025-06-04 20:14:39                20                        2           0   
9 2025-06-05 00:44:47                 0                        3           0   

   hour_sin  hour_cos  user_tenure_days  content_age_days  freshness  
0 -0.258819  0.965926        204.085683         

## 14. Point-in-Time User Activity Features

User activity features summarize how active a user has been before
the current interaction.

Only historical interactions are used so that these features do not
leak information from the current or future interactions.

In [89]:
# Restore chronological order within each user
model_data = model_data.sort_values(
    ["user_id", "timestamp"]
).copy()

# Number of previous interactions already exists,
# but we recompute it here to keep this feature block self-contained.
user_group = model_data.groupby("user_id")

model_data["user_prior_interactions"] = (
    user_group.cumcount()
)

# Create a calendar-day representation for activity tracking
model_data["interaction_date"] = (
    model_data["timestamp"].dt.date
)

# Historical active days
user_date_group = model_data.groupby(
    ["user_id", "interaction_date"]
)

model_data["user_date_interaction_number"] = (
    user_date_group.cumcount()
)

# A user's first interaction on a given date represents
# one new historical active day.
model_data["is_first_interaction_of_day"] = (
    model_data["user_date_interaction_number"] == 0
).astype(int)

model_data["user_prior_active_days"] = (
    model_data.groupby("user_id")[
        "is_first_interaction_of_day"
    ].cumsum()
    - model_data["is_first_interaction_of_day"]
)

# Historical unique content count
model_data["user_content_seen_before"] = (
    model_data.groupby(
        ["user_id", "content_id"]
    ).cumcount()
)

model_data["user_prior_unique_content"] = (
    model_data.groupby("user_id")[
        "user_content_seen_before"
    ].transform(
        lambda x: (x > 0).cumsum()
    )
)

# Historical unique creator count
model_data["user_creator_seen_before"] = (
    model_data.groupby(
        ["user_id", "creator_id"]
    ).cumcount()
)

model_data["user_prior_unique_creators"] = (
    model_data.groupby("user_id")[
        "user_creator_seen_before"
    ].transform(
        lambda x: (x > 0).cumsum()
    )
)

print(
    model_data[
        [
            "user_id",
            "timestamp",
            "user_prior_interactions",
            "user_prior_active_days",
            "user_prior_unique_content",
            "user_prior_unique_creators"
        ]
    ].head(20)
)

      user_id           timestamp  user_prior_interactions  \
5027   U00001 2025-08-29 04:40:07                        0   
13463  U00001 2025-10-20 17:40:08                        1   
15271  U00001 2025-10-29 11:12:49                        2   
21293  U00001 2025-11-23 05:33:29                        3   
25966  U00001 2025-12-09 15:59:55                        4   
29179  U00001 2025-12-19 22:27:29                        5   
36454  U00001 2026-01-10 00:24:10                        6   
39062  U00001 2026-01-17 00:08:40                        7   
50469  U00001 2026-02-13 14:19:29                        8   
50696  U00001 2026-02-14 02:30:09                        9   
52263  U00001 2026-02-17 13:00:08                       10   
52371  U00001 2026-02-17 18:41:53                       11   
54020  U00001 2026-02-21 06:11:45                       12   
54362  U00001 2026-02-22 00:29:04                       13   
56270  U00001 2026-02-25 21:08:33                       14   
59489  U

In [90]:
# Validate that the first interaction for every user has zero history
first_user_interactions = (
    model_data
    .sort_values(["user_id", "timestamp"])
    .groupby("user_id")
    .first()
)

assert (
    first_user_interactions["user_prior_interactions"] == 0
).all()

assert (
    first_user_interactions["user_prior_active_days"] == 0
).all()

assert (
    first_user_interactions["user_prior_unique_content"] == 0
).all()

assert (
    first_user_interactions["user_prior_unique_creators"] == 0
).all()

print("✓ First-interaction history checks passed")


# Validate that unique content never decreases
content_check = (
    model_data
    .sort_values(["user_id", "timestamp"])
    .groupby("user_id")["user_prior_unique_content"]
    .diff()
    .dropna()
)

assert (content_check >= 0).all()

print("✓ Unique-content history is monotonic")


# Validate that unique creators never decreases
creator_check = (
    model_data
    .sort_values(["user_id", "timestamp"])
    .groupby("user_id")["user_prior_unique_creators"]
    .diff()
    .dropna()
)

assert (creator_check >= 0).all()

print("✓ Unique-creator history is monotonic")

✓ First-interaction history checks passed
✓ Unique-content history is monotonic
✓ Unique-creator history is monotonic


In [92]:
print(users.columns.tolist())

['user_id', 'country', 'age_group', 'signup_date', 'following_count', 'creator_flag', 'preferred_genres']


In [93]:
user_static_features = users[
    [
        "user_id",
        "age_group",
        "country",
        "following_count",
        "creator_flag"
    ]
].copy()

user_static_features.head()

,user_id,age_group,country,following_count,creator_flag
0,U00001,13-17,India,15,True
1,U00002,25-34,Germany,17,False
2,U00003,45+,Australia,20,False
3,U00004,35-44,Canada,13,False
4,U00005,18-24,Canada,24,False


In [94]:
print("Shape:", user_static_features.shape)

print("\nData types:")
print(user_static_features.dtypes)

print("\nMissing values:")
print(user_static_features.isna().sum())

print("\nAge groups:")
print(user_static_features["age_group"].value_counts())

print("\nCreator flag:")
print(user_static_features["creator_flag"].value_counts())

print("\nFollowing count:")
print(user_static_features["following_count"].describe())

Shape: (5000, 5)

Data types:
user_id            object
age_group          object
country            object
following_count     int64
creator_flag         bool
dtype: object

Missing values:
user_id            0
age_group          0
country            0
following_count    0
creator_flag       0
dtype: int64

Age groups:
age_group
18-24    1785
25-34    1520
35-44     973
45+       464
13-17     258
Name: count, dtype: int64

Creator flag:
creator_flag
False    4587
True      413
Name: count, dtype: int64

Following count:
count    5000.000000
mean       20.073800
std         4.485664
min         5.000000
25%        17.000000
50%        20.000000
75%        23.000000
max        38.000000
Name: following_count, dtype: float64


In [95]:
content_static_features = content[
    [
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration"
    ]
].copy()

content_static_features = content_static_features.merge(
    creators[
        [
            "creator_id",
            "followers",
            "creator_type"
        ]
    ],
    on="creator_id",
    how="left"
)

content_static_features = content_static_features.rename(
    columns={
        "followers": "creator_followers"
    }
)

content_static_features.head()

,content_id,creator_id,genre,content_type,duration,creator_followers,creator_type
0,CT000001,C0131,Mystery,mini,24.6,715,AI Creator
1,CT000002,C0484,Horror,mini,21.6,291,Community Creator
2,CT000003,C0195,Sci-Fi,mini,14.7,33,Artist
3,CT000004,C0328,Romance,mini,6.6,49,Filmmaker
4,CT000005,C0406,Action,image,88.7,74,AI Creator


In [96]:
print("Shape:", content_static_features.shape)

print("\nData types:")
print(content_static_features.dtypes)

print("\nMissing values:")
print(content_static_features.isna().sum())

print("\nContent types:")
print(content_static_features["content_type"].value_counts())

print("\nGenres:")
print(content_static_features["genre"].value_counts())

print("\nCreator types:")
print(content_static_features["creator_type"].value_counts())

print("\nDuration:")
print(content_static_features["duration"].describe())

print("\nCreator followers:")
print(content_static_features["creator_followers"].describe())

Shape: (10000, 7)

Data types:
content_id            object
creator_id            object
genre                 object
content_type          object
duration             float64
creator_followers      int64
creator_type          object
dtype: object

Missing values:
content_id           0
creator_id           0
genre                0
content_type         0
duration             0
creator_followers    0
creator_type         0
dtype: int64

Content types:
content_type
mini     4524
image    2023
cine     1941
video    1512
Name: count, dtype: int64

Genres:
genre
Horror         1040
Romance        1024
Action         1014
Drama          1013
Documentary    1012
Mystery         996
Comedy          993
Animation       990
Fantasy         970
Sci-Fi          948
Name: count, dtype: int64

Creator types:
creator_type
AI Creator           2286
Artist               2278
Influencer           1900
Filmmaker            1795
Community Creator    1741
Name: count, dtype: int64

Duration:
count    1000

In [97]:
print("model_data shape:", model_data.shape)

print("\nCurrent columns:")
for i, col in enumerate(model_data.columns, 1):
    print(f"{i:2d}. {col}")

model_data shape: (250000, 70)

Current columns:
 1. user_id
 2. content_id
 3. creator_id
 4. genre
 5. content_type
 6. duration
 7. creator_followers
 8. content_created_at
 9. genre_match
10. timestamp
11. content_age_days
12. freshness
13. impression
14. clicked
15. watch_time
16. completion_rate
17. liked
18. saved
19. shared
20. commented
21. recreated
22. meaningful_engagement
23. user_prior_interactions
24. user_prior_clicks
25. user_prior_engagements
26. user_prior_watch_time
27. user_prior_completions
28. user_prior_ctr
29. user_prior_engagement_rate
30. user_prior_avg_watch_time
31. user_prior_avg_completion
32. creator_prior_interactions
33. creator_prior_clicks
34. creator_prior_engagements
35. creator_prior_ctr
36. creator_prior_engagement_rate
37. content_prior_interactions
38. content_prior_clicks
39. content_prior_engagements
40. content_prior_ctr
41. content_prior_engagement_rate
42. user_content_prior_interactions
43. user_content_prior_clicks
44. user_content_prior

In [98]:
# Columns that identify entities or timestamps
identifier_columns = [
    "user_id",
    "content_id",
    "creator_id",
    "timestamp",
    "content_created_at",
    "signup_date",
]

# Current interaction outcomes / target
outcome_columns = [
    "impression",
    "clicked",
    "watch_time",
    "completion_rate",
    "liked",
    "saved",
    "shared",
    "commented",
    "recreated",
    "meaningful_engagement",
]

# Features we know are derived from information available before
# the current interaction
historical_feature_keywords = [
    "prior",
    "freshness",
    "content_age",
    "tenure",
    "hour",
    "day_of_week",
    "weekend",
    "hour_sin",
    "hour_cos",
]

print("Identifier / metadata columns:")
print(identifier_columns)

print("\nOutcome / target columns:")
print(outcome_columns)

print("\nPotential historical/context features:")
for col in model_data.columns:
    if any(keyword in col for keyword in historical_feature_keywords):
        print(col)

Identifier / metadata columns:
['user_id', 'content_id', 'creator_id', 'timestamp', 'content_created_at', 'signup_date']

Outcome / target columns:
['impression', 'clicked', 'watch_time', 'completion_rate', 'liked', 'saved', 'shared', 'commented', 'recreated', 'meaningful_engagement']

Potential historical/context features:
content_age_days
freshness
user_prior_interactions
user_prior_clicks
user_prior_engagements
user_prior_watch_time
user_prior_completions
user_prior_ctr
user_prior_engagement_rate
user_prior_avg_watch_time
user_prior_avg_completion
creator_prior_interactions
creator_prior_clicks
creator_prior_engagements
creator_prior_ctr
creator_prior_engagement_rate
content_prior_interactions
content_prior_clicks
content_prior_engagements
content_prior_ctr
content_prior_engagement_rate
user_content_prior_interactions
user_content_prior_clicks
user_content_prior_engagements
user_content_prior_ctr
user_content_prior_engagement_rate
user_genre_prior_interactions
user_genre_prior_click

In [99]:
feature_columns = [
    # User static
    "age_group",
    "country",
    "following_count",
    "creator_flag",

    # Content / creator static
    "genre",
    "content_type",
    "duration",
    "creator_followers",
    "creator_type",

    # Current context
    "content_age_days",
    "freshness",
    "interaction_hour",
    "interaction_day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "user_tenure_days",

    # User history
    "user_prior_interactions",
    "user_prior_clicks",
    "user_prior_engagements",
    "user_prior_watch_time",
    "user_prior_completions",
    "user_prior_ctr",
    "user_prior_engagement_rate",
    "user_prior_avg_watch_time",
    "user_prior_avg_completion",
    "user_prior_active_days",
    "user_prior_unique_content",
    "user_prior_unique_creators",

    # Creator history
    "creator_prior_interactions",
    "creator_prior_clicks",
    "creator_prior_engagements",
    "creator_prior_ctr",
    "creator_prior_engagement_rate",

    # Content history
    "content_prior_interactions",
    "content_prior_clicks",
    "content_prior_engagements",
    "content_prior_ctr",
    "content_prior_engagement_rate",

    # User-content relationship
    "user_content_prior_interactions",
    "user_content_prior_clicks",
    "user_content_prior_engagements",
    "user_content_prior_ctr",
    "user_content_prior_engagement_rate",

    # User-genre relationship
    "user_genre_prior_interactions",
    "user_genre_prior_clicks",
    "user_genre_prior_engagements",
    "user_genre_prior_engagement_rate",
]

print("Number of candidate features:", len(feature_columns))

missing_features = [
    col for col in feature_columns
    if col not in model_data.columns
]

print("\nMissing features:", missing_features)

Number of candidate features: 48

Missing features: ['age_group', 'country', 'following_count', 'creator_flag', 'creator_type']


In [100]:
model_data = model_data.merge(
    user_static_features,
    on="user_id",
    how="left",
    validate="many_to_one"
)

print("After user feature merge:", model_data.shape)

After user feature merge: (250000, 74)


In [101]:
model_data = model_data.merge(
    content_static_features,
    on=["content_id", "creator_id"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_static")
)

print("After content feature merge:", model_data.shape)

After content feature merge: (250000, 79)


In [102]:
# Check that the static features were successfully attached
static_columns = [
    "age_group",
    "country",
    "following_count",
    "creator_flag",
    "creator_type"
]

print("Missing static features:")
print(model_data[static_columns].isna().sum())

print("\nRow count:", len(model_data))

print("\nUnique interactions:")
print(model_data[["user_id", "content_id", "timestamp"]].drop_duplicates().shape[0])

print("\nExpected interactions:", len(interactions))

Missing static features:
age_group          0
country            0
following_count    0
creator_flag       0
creator_type       0
dtype: int64

Row count: 250000

Unique interactions:
250000

Expected interactions: 250000


In [103]:
assert len(model_data) == len(interactions)

assert model_data["user_id"].notna().all()
assert model_data["content_id"].notna().all()
assert model_data["creator_id"].notna().all()

assert model_data[static_columns].notna().all().all()

print("✓ Row count preserved")
print("✓ Required IDs present")
print("✓ Static features successfully attached")
print("✓ Join integrity passed")

✓ Row count preserved
✓ Required IDs present
✓ Static features successfully attached
✓ Join integrity passed


In [104]:
# ==========================================
# LEAKAGE & MISSING-VALUE AUDIT
# ==========================================

# 1. Check that none of our candidate features are outcome columns
leakage_columns = [
    col for col in feature_columns
    if col in outcome_columns
]

print("Potential leakage columns:")
print(leakage_columns)


# 2. Missing-value summary for candidate features
missing_summary = (
    model_data[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

print("\nFeatures with missing values:")
print(missing_summary)


# 3. Percentage missing
if len(missing_summary) > 0:
    missing_percentage = (
        model_data[missing_summary.index]
        .isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )

    print("\nMissing percentage:")
    print(missing_percentage)
else:
    print("\nNo missing values found.")

Potential leakage columns:
[]

Features with missing values:
user_content_prior_ctr                248706
user_content_prior_engagement_rate    248706
user_genre_prior_engagement_rate       45724
content_prior_engagement_rate          10000
content_prior_ctr                      10000
user_prior_engagement_rate              4999
user_prior_ctr                          4999
user_prior_avg_completion               4999
user_prior_avg_watch_time               4999
creator_prior_ctr                        500
creator_prior_engagement_rate            500
dtype: int64

Missing percentage:
user_content_prior_ctr                99.4824
user_content_prior_engagement_rate    99.4824
user_genre_prior_engagement_rate      18.2896
content_prior_engagement_rate          4.0000
content_prior_ctr                      4.0000
user_prior_engagement_rate             1.9996
user_prior_ctr                         1.9996
user_prior_avg_completion              1.9996
user_prior_avg_watch_time              1.9

In [105]:
# ==========================================
# COLD-START / HISTORY COVERAGE AUDIT
# ==========================================

history_features = {
    "user": "user_prior_interactions",
    "content": "content_prior_interactions",
    "creator": "creator_prior_interactions",
    "user_content": "user_content_prior_interactions",
    "user_genre": "user_genre_prior_interactions",
}

for entity, column in history_features.items():
    has_history = model_data[column] > 0

    print(
        f"{entity:15s} | "
        f"has history: {has_history.mean() * 100:6.2f}% | "
        f"cold start: {(~has_history).mean() * 100:6.2f}%"
    )

user            | has history:  98.00% | cold start:   2.00%
content         | has history:  96.00% | cold start:   4.00%
creator         | has history:  99.80% | cold start:   0.20%
user_content    | has history:   0.52% | cold start:  99.48%
user_genre      | has history:  81.71% | cold start:  18.29%


In [106]:
# ==========================================
# COLD-START INDICATORS
# ==========================================

model_data["user_has_history"] = (
    model_data["user_prior_interactions"] > 0
).astype(int)

model_data["content_has_history"] = (
    model_data["content_prior_interactions"] > 0
).astype(int)

model_data["creator_has_history"] = (
    model_data["creator_prior_interactions"] > 0
).astype(int)

model_data["user_content_has_history"] = (
    model_data["user_content_prior_interactions"] > 0
).astype(int)

model_data["user_genre_has_history"] = (
    model_data["user_genre_prior_interactions"] > 0
).astype(int)

In [107]:
cold_start_features = [
    "user_has_history",
    "content_has_history",
    "creator_has_history",
    "user_content_has_history",
    "user_genre_has_history"
]

print(
    model_data[cold_start_features]
    .mean()
    .mul(100)
    .round(2)
)

user_has_history            98.00
content_has_history         96.00
creator_has_history         99.80
user_content_has_history     0.52
user_genre_has_history      81.71
dtype: float64


The formula:

$$ \text{smoothed rate} = \frac{s+\alpha p}{n+\alpha} $$

where:

\(s\) = prior successes
\(n\) = prior interactions
\(p\) = global engagement rate
\(\alpha\) = strength of the prior

We'll use alpha = 10 initially. Later, we can tune this rather than pretending 10 is magically optimal.

In [108]:
# ==========================================
# SMOOTHED HISTORICAL RATES
# ==========================================

global_engagement_rate = model_data["meaningful_engagement"].mean()

alpha = 10

print("Global engagement rate:", round(global_engagement_rate, 4))
print("Smoothing strength (alpha):", alpha)


def smoothed_rate(successes, interactions):
    return (
        successes + alpha * global_engagement_rate
    ) / (
        interactions + alpha
    )


# User
model_data["user_prior_smoothed_engagement_rate"] = smoothed_rate(
    model_data["user_prior_engagements"],
    model_data["user_prior_interactions"]
)


# Creator
model_data["creator_prior_smoothed_engagement_rate"] = smoothed_rate(
    model_data["creator_prior_engagements"],
    model_data["creator_prior_interactions"]
)


# Content
model_data["content_prior_smoothed_engagement_rate"] = smoothed_rate(
    model_data["content_prior_engagements"],
    model_data["content_prior_interactions"]
)


# User-content
model_data["user_content_prior_smoothed_engagement_rate"] = smoothed_rate(
    model_data["user_content_prior_engagements"],
    model_data["user_content_prior_interactions"]
)


# User-genre
model_data["user_genre_prior_smoothed_engagement_rate"] = smoothed_rate(
    model_data["user_genre_prior_engagements"],
    model_data["user_genre_prior_interactions"]
)

print("\n✓ Smoothed historical rates created")

Global engagement rate: 0.3559
Smoothing strength (alpha): 10

✓ Smoothed historical rates created


In [109]:
# ==========================================
# VALIDATE SMOOTHED RATES
# ==========================================

rate_pairs = [
    (
        "user",
        "user_prior_interactions",
        "user_prior_engagements",
        "user_prior_smoothed_engagement_rate"
    ),
    (
        "creator",
        "creator_prior_interactions",
        "creator_prior_engagements",
        "creator_prior_smoothed_engagement_rate"
    ),
    (
        "content",
        "content_prior_interactions",
        "content_prior_engagements",
        "content_prior_smoothed_engagement_rate"
    ),
    (
        "user-content",
        "user_content_prior_interactions",
        "user_content_prior_engagements",
        "user_content_prior_smoothed_engagement_rate"
    ),
    (
        "user-genre",
        "user_genre_prior_interactions",
        "user_genre_prior_engagements",
        "user_genre_prior_smoothed_engagement_rate"
    ),
]

for name, interactions_col, successes_col, smoothed_col in rate_pairs:
    cold = model_data[interactions_col] == 0

    cold_values = model_data.loc[cold, smoothed_col]

    print(f"\n{name}")
    print(f"  Cold-start rows: {cold.sum():,}")
    print(
        f"  Cold-start smoothed rate: "
        f"{cold_values.mean():.4f}"
    )

    # Smoothed rates must always be valid probabilities
    assert model_data[smoothed_col].between(0, 1).all()

print("\n✓ All smoothed rates are valid probabilities")


user
  Cold-start rows: 4,999
  Cold-start smoothed rate: 0.3559

creator
  Cold-start rows: 500
  Cold-start smoothed rate: 0.3559

content
  Cold-start rows: 10,000
  Cold-start smoothed rate: 0.3559

user-content
  Cold-start rows: 248,706
  Cold-start smoothed rate: 0.3559

user-genre
  Cold-start rows: 45,724
  Cold-start smoothed rate: 0.3559

✓ All smoothed rates are valid probabilities


In [110]:
smoothed_rate_features = [
    "user_prior_smoothed_engagement_rate",
    "creator_prior_smoothed_engagement_rate",
    "content_prior_smoothed_engagement_rate",
    "user_content_prior_smoothed_engagement_rate",
    "user_genre_prior_smoothed_engagement_rate",
]

feature_columns.extend(smoothed_rate_features)

print("Total candidate features:", len(feature_columns))

print("\nSmoothed features added:")
for feature in smoothed_rate_features:
    print("-", feature)

Total candidate features: 53

Smoothed features added:
- user_prior_smoothed_engagement_rate
- creator_prior_smoothed_engagement_rate
- content_prior_smoothed_engagement_rate
- user_content_prior_smoothed_engagement_rate
- user_genre_prior_smoothed_engagement_rate


In [111]:
missing_features = [
    col for col in feature_columns
    if col not in model_data.columns
]

print("Missing features:", missing_features)

assert len(missing_features) == 0

print("✓ All candidate features exist")

Missing features: []
✓ All candidate features exist


In [112]:
# ==========================================
# FINAL MODEL DATASET
# ==========================================

target_column = "meaningful_engagement"

X = model_data[feature_columns].copy()
y = model_data[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

X shape: (250000, 53)
y shape: (250000,)

Target distribution:
meaningful_engagement
0    161019
1     88981
Name: count, dtype: int64

Target percentage:
meaningful_engagement
0    64.41
1    35.59
Name: proportion, dtype: float64


In [113]:
# ==========================================
# FINAL LEAKAGE AUDIT
# ==========================================

# Outcome columns must not be present in X
outcome_in_features = [
    col for col in outcome_columns
    if col in X.columns
]

print("Outcome columns found in X:")
print(outcome_in_features)

assert len(outcome_in_features) == 0


# Synthetic oracle feature must not be present
assert "genre_match" not in X.columns

print("✓ No outcome variables in X")
print("✓ genre_match oracle feature excluded")


# IDs must not be predictive model features
id_features = [
    "user_id",
    "content_id",
    "creator_id",
    "timestamp",
    "content_created_at",
    "signup_date",
]

ids_in_features = [
    col for col in id_features
    if col in X.columns
]

print("\nIDs/timestamps found in X:")
print(ids_in_features)

assert len(ids_in_features) == 0

print("✓ IDs and raw timestamps excluded")

Outcome columns found in X:
[]
✓ No outcome variables in X
✓ genre_match oracle feature excluded

IDs/timestamps found in X:
[]
✓ IDs and raw timestamps excluded


In [114]:
print("Numerical features:")
numeric_features = X.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

print(len(numeric_features))
print(numeric_features)

print("\nCategorical features:")
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(len(categorical_features))
print(categorical_features)

Numerical features:
46
['following_count', 'creator_flag', 'duration', 'creator_followers', 'content_age_days', 'freshness', 'is_weekend', 'hour_sin', 'hour_cos', 'user_tenure_days', 'user_prior_interactions', 'user_prior_clicks', 'user_prior_engagements', 'user_prior_watch_time', 'user_prior_completions', 'user_prior_ctr', 'user_prior_engagement_rate', 'user_prior_avg_watch_time', 'user_prior_avg_completion', 'user_prior_active_days', 'user_prior_unique_content', 'user_prior_unique_creators', 'creator_prior_interactions', 'creator_prior_clicks', 'creator_prior_engagements', 'creator_prior_ctr', 'creator_prior_engagement_rate', 'content_prior_interactions', 'content_prior_clicks', 'content_prior_engagements', 'content_prior_ctr', 'content_prior_engagement_rate', 'user_content_prior_interactions', 'user_content_prior_clicks', 'user_content_prior_engagements', 'user_content_prior_ctr', 'user_content_prior_engagement_rate', 'user_genre_prior_interactions', 'user_genre_prior_clicks', 'user

In [115]:
# ==========================================
# FINAL TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ==========================================

model_data = model_data.sort_values("timestamp").reset_index(drop=True)

n = len(model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = model_data.iloc[:train_end].copy()
val_data = model_data.iloc[train_end:val_end].copy()
test_data = model_data.iloc[val_end:].copy()

print("Dataset sizes:")
print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

print("\nTime ranges:")

print(
    "Train:",
    train_data["timestamp"].min(),
    "→",
    train_data["timestamp"].max()
)

print(
    "Validation:",
    val_data["timestamp"].min(),
    "→",
    val_data["timestamp"].max()
)

print(
    "Test:",
    test_data["timestamp"].min(),
    "→",
    test_data["timestamp"].max()
)

# Temporal assertions
assert train_data["timestamp"].max() <= val_data["timestamp"].min()
assert val_data["timestamp"].max() <= test_data["timestamp"].min()

print("\n✓ Temporal ordering verified")
print("✓ No future observations appear in earlier splits")

Dataset sizes:
Train: (175000, 89)
Validation: (37500, 89)
Test: (37500, 89)

Time ranges:
Train: 2025-06-01 23:55:16 → 2026-07-22 07:49:04
Validation: 2026-07-22 07:50:29 → 2026-08-15 05:05:32
Test: 2026-08-15 05:05:41 → 2026-08-30 23:59:46

✓ Temporal ordering verified
✓ No future observations appear in earlier splits


In [116]:
# ==========================================
# FEATURE SET — FROZEN
# ==========================================

FINAL_FEATURE_COLUMNS = feature_columns.copy()

print("FINAL FEATURE COUNT:", len(FINAL_FEATURE_COLUMNS))
print("\nFinal feature list:")

for i, feature in enumerate(FINAL_FEATURE_COLUMNS, start=1):
    print(f"{i:02d}. {feature}")

print("\n✓ Feature set frozen for modeling")

FINAL FEATURE COUNT: 53

Final feature list:
01. age_group
02. country
03. following_count
04. creator_flag
05. genre
06. content_type
07. duration
08. creator_followers
09. creator_type
10. content_age_days
11. freshness
12. interaction_hour
13. interaction_day_of_week
14. is_weekend
15. hour_sin
16. hour_cos
17. user_tenure_days
18. user_prior_interactions
19. user_prior_clicks
20. user_prior_engagements
21. user_prior_watch_time
22. user_prior_completions
23. user_prior_ctr
24. user_prior_engagement_rate
25. user_prior_avg_watch_time
26. user_prior_avg_completion
27. user_prior_active_days
28. user_prior_unique_content
29. user_prior_unique_creators
30. creator_prior_interactions
31. creator_prior_clicks
32. creator_prior_engagements
33. creator_prior_ctr
34. creator_prior_engagement_rate
35. content_prior_interactions
36. content_prior_clicks
37. content_prior_engagements
38. content_prior_ctr
39. content_prior_engagement_rate
40. user_content_prior_interactions
41. user_content_pr

In [117]:
import json
from pathlib import Path

feature_path = Path("../data/processed/final_feature_columns.json")

with open(feature_path, "w") as f:
    json.dump(FINAL_FEATURE_COLUMNS, f, indent=2)

print(f"Saved feature specification to: {feature_path}")

Saved feature specification to: ..\data\processed\final_feature_columns.json


In [118]:
# ==========================================
# SAVE FINAL FEATURE-ENGINEERED DATASET
# ==========================================

model_data_path = Path("../data/processed/model_features.csv")

model_data.to_csv(
    model_data_path,
    index=False
)

print(f"Saved feature-engineered dataset to: {model_data_path}")
print("Shape:", model_data.shape)

Saved feature-engineered dataset to: ..\data\processed\model_features.csv
Shape: (250000, 89)
